In [4]:
import numpy as np
import pickle
from attogpt.tokenizer import BPETokenizer
from attogpt.utils import softmax

with open("../data/plato/checkpts/checkpt.pkl", "rb") as f:
    checkpt = pickle.load(f)

model = checkpt["model"]

rng = np.random.default_rng(model.config.seed)

tokenizer = BPETokenizer(model.config)
tokenizer.from_file("../data/plato/vocabulary.json")

T = model.config.seq_len


In [ ]:
n_toks = 50
temperature = 1.0

prompt = input("Seed: ")
ids = tokenizer.encode(prompt)

print(tokenizer.decode(ids), end="")

for _ in range(n_toks):
    if len(ids) >= T:
        window = np.asarray(ids[-T:], dtype=np.int64)[None, :]
        t = T - 1
    else:
        window = np.zeros((1, T), dtype=np.int64)
        window[0, :len(ids)] = ids
        t = len(ids) - 1
    
    logits = model.forward(window)
    probs = softmax(logits[0, t], temperature=temperature)
    next = int(rng.choice(len(probs), p=probs))
    ids.append(next)
    print(tokenizer.decode([next]), end="", flush=True)
